# 05 - Category, Region, Seller and Seasonality
**Questions:** which categories and states drive revenue and growth, which sellers underperform, and what does seasonality look like?

Sources: `marts.mart_category_performance`, `marts.mart_state_performance`, `marts.mart_seller_performance`, `marts.metrics_kpis`.

In [ ]:
import sys
from pathlib import Path

import duckdb
import pandas as pd
import plotly.express as px

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from ingestion.config import get_warehouse_path  # noqa: E402

con = duckdb.connect(str(get_warehouse_path()), read_only=True)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")


def q(sql: str) -> pd.DataFrame:
    """Run SQL against the warehouse and return a DataFrame."""
    return con.execute(sql).df()

## Revenue trend and seasonality

In [ ]:
kpis = q("SELECT * FROM marts.metrics_kpis ORDER BY year_month")
display(kpis)
px.line(kpis, x="year_month", y="delivered_revenue", markers=True, title="Delivered revenue by month (BRL)").show()
px.line(kpis, x="year_month", y=["on_time_pct"], markers=True, title="On-time delivery % by month").show()

In [ ]:
dow = q('''
    SELECT d.day_name, d.day_of_week, COUNT(*) AS delivered_orders, SUM(f.delivered_revenue) AS revenue
    FROM marts.fact_orders f JOIN marts.dim_date d ON f.order_date = d.date_day
    WHERE f.in_analysis_window AND f.is_delivered
    GROUP BY 1, 2 ORDER BY 2
''')
display(dow)
px.bar(dow, x="day_name", y="delivered_orders", title="Delivered orders by day of week").show()

Note on seasonality: the window holds only one holiday season, so check whether a single spike (such as November) drives the pattern before calling it seasonal.

## Categories

In [ ]:
cat = q("SELECT * FROM marts.mart_category_performance ORDER BY revenue_rank")
display(cat.head(15))
px.bar(cat.head(15), x="category", y="revenue", title="Top 15 categories by revenue (BRL)").show()
n80 = int((cat["cumulative_share_pct"] < 80).sum()) + 1
print(f"{n80} of {len(cat)} categories account for about 80% of revenue.")

### Growth (year over year, same calendar dates)
Small categories can show huge percentage growth from a tiny base, so only categories with a meaningful prior-period share are shown.

In [ ]:
MIN_PRIOR_SHARE_PCT = 1.0   # ignore categories below this share of prior-period revenue
total_prior = cat["revenue_prior_period"].astype(float).sum()
cat["prior_share_pct"] = 100 * cat["revenue_prior_period"].astype(float) / total_prior
growth = cat[cat["prior_share_pct"] >= MIN_PRIOR_SHARE_PCT].sort_values("yoy_growth_pct", ascending=False)
display(growth[["category", "revenue_prior_period", "revenue_current_period", "yoy_growth_pct", "prior_share_pct"]])
px.bar(growth, x="category", y="yoy_growth_pct", title="Year-over-year growth, categories above the share threshold").show()

## States

In [ ]:
st = q("SELECT * FROM marts.mart_state_performance ORDER BY delivered_revenue DESC")
display(st.head(10))
px.bar(st, x="state", y="delivered_revenue", title="Delivered revenue by customer state (BRL)").show()
display(st.sort_values("on_time_pct").head(8)[["state", "delivered_orders", "on_time_pct", "avg_delivery_days", "avg_review_score"]])

## Sellers

In [ ]:
sel = q("SELECT * FROM marts.mart_seller_performance")
qual = sel[sel["meets_min_volume"]].copy()
print(f"{len(qual)} of {len(sel)} sellers meet the minimum volume and are rankable.")
print(f"Marketplace benchmarks: late % = {sel['overall_late_pct'].iloc[0]}, avg review = {sel['overall_avg_review_score'].iloc[0]}")
display(qual.sort_values("avg_review_score").head(10)[["seller_key", "state", "orders", "revenue", "late_pct", "avg_review_score"]])
display(qual.sort_values("late_pct", ascending=False).head(10)[["seller_key", "state", "orders", "revenue", "late_pct", "avg_review_score"]])

## How to read this
- Seller metrics attribute multi-seller orders to every seller involved, so treat them as indicative.
- Always check order volume before calling a seller a problem: a handful of late orders from a small seller is noise.
- Record top categories, growth leaders, weakest regions and the weakest high-volume sellers in `docs/insights.md`.